# Elemental composition

Here we have a code to compute the elemental formula from a given mass 
<br>
note that this rely on <br>
    1. assuming the mass is accurate <br>
    2. your given element restriction is correct (i.e., no S or halogens)<br>
    3. this also assumes that the m/z value is from adding or removing proton (i.e., M+H or M-H), you could throw in neutral mass and make the "mode" parameter to be "neutral"<br>
    4. use nitrogen rule and DBE filter on your own risk, as some higher mass species can be problematic<br>
    5. this will try to match Na if M+H fails. this is not the right approach but better than nothing lol<br>


use could put S in the possible element, though this function DOES NOT check isotope (as it's purely matching through given m/z value). so suggested approach is to compute without S first then with S again after. 



how to use:

for each m/z value in your data, iterate through each, do whatever checks you would like, and if you need to compute the formula, call this function 


this function return two values, the best formula (decided by the minimum mass error) and possible neutral form mass of the target

note tha the calcualted neutral mass is likely slightly off from the real theo. mass since the target mass will be slightly off

In [1]:
from molmass import Formula
from pyopenms import ElementDB
import numpy as np

In [2]:

def compute_formula_from_mass_efficient(target, tol=5.0, tol_unit="ppm", mode="positive", elements=None,
                                         apply_nitrogen_rule=False, apply_hc_ratio=False,
                                         apply_rdbe_filter=False, rdbe_min=0.0, rdbe_max=40.0,
                                         apply_ratio_filter=False, oc_max=1.2, nc_max=1.3,
                                         return_tuple=True, no_match_return = "NA"):
    """
    Computes molecular formula from a given mass with high efficiency using dynamic nested loops (recursion) 
    and fast calculation for the lightest element.
    
    Args:
        target (float): The input m/z value.
        tol (float): Tolerance value.
        tol_unit (str): Unit of tolerance ("ppm" or "Da").
        mode (str): Ionization mode ("positive", "pos", "negative", "neg").
        elements (dict): Dictionary of element limits. Default: {'C': 50, 'H': 100, 'N': 10, 'O': 20, 'S': 2}.
                         This defines BOTH the allowed elements and their maximum counts.
        apply_nitrogen_rule (bool): Whether to apply the Nitrogen Rule (valid for mass < 500).
        apply_hc_ratio (bool): Whether to enforce H/C <= 4.
        apply_rdbe_filter (bool): Whether to enforce ring+double-bond equivalent bounds.
            RDBE = C - H/2 + N/2 + 1 (ignores O, S; would need -X/2 term if halogens added).
            Rejects negative RDBE (chemically impossible) and anything above rdbe_max.
        rdbe_min (float): Minimum allowed RDBE. Default 0.0 (no negative RDBE allowed).
        rdbe_max (float): Maximum allowed RDBE. Default 40.0 — loosen/tighten for your mass range.
        apply_ratio_filter (bool): Whether to enforce O/C and N/C heteroatom ratio bounds.
            Defaults follow the general heuristic bounds discussed in Kind & Fiehn's
            "seven golden rules" formula-validation approach — verify this citation
            and whether these specific bounds suit BBOA chemistry before relying on them.
        oc_max (float): Maximum O/C ratio. Default 1.2.
        nc_max (float): Maximum N/C ratio. Default 1.3 — you may want this much tighter
            (e.g. 0.3-0.5) given real BBOA nitrogen content is typically low; the literature
            default is tuned for general small-molecule metabolomics, not combustion aerosol.

    Returns:
        tuple or str or None: The (formula string, mass) of the best match, or "NA" if no match found.
    """
    
    # 1. Setup Elements and Masses n 
    edb = ElementDB()
    default_limits = {'C': 50, 'H': 100, 'N': 10, 'O': 20}
    
    if elements is None:
        limits = default_limits
    else:
        limits = elements
        
    # Validation and Pre-processing
    element_data = [] # List of dicts: {'el': symbol, 'mass': mass, 'limit': limit}
    
    try:
        for el, limit in limits.items():
            try:
                msg = edb.getElement(el)
                mono_mass = msg.getMonoWeight()
                nom_mass =  Formula(el).nominal_mass
            except:
                raise ValueError(f"Invalid element symbol: {el}")
            
            element_data.append({
                'el': el,
                'mass': mono_mass,
                'limit': limit,
                'nom_mass': nom_mass
            })
    except ValueError as e:
        print(f"Error initializing elements: {e}")
        return "Error for elements"

    if not element_data:
        print("No elements provided.")
        return "NA"

    # Strategy: 
    # 1. Identify "Calculation Element" -> The lightest one (usually H).
    #    We calculate this directly instead of iterating to save 100x iterations.
    # 2. Identify "Iteration Elements" -> The rest.
    #    Sort them by mass Descending (Heaviest first).
    #    This allows early pruning in the recursion (breaking early if mass exceeded).
    
    # Find lightest
    element_data.sort(key=lambda x: x['mass'])
    calc_element = element_data[0] # Lightest
    iteration_elements = element_data[1:]
    
    # Sort iteration elements by mass Descending
    iteration_elements.sort(key=lambda x: x['mass'], reverse=True)
    
    # Pre-calculate constants
    mass_proton = 1.0072764666
    mass_electron = 0.00054858
    try:
        mass_Na = edb.getElement("Na").getMonoWeight()
    except:
        mass_Na = 22.989769 # Fallback if Na not in user list logic? No, independent lookup.
        
    calc_el_mass = calc_element['mass']
    calc_el_limit = calc_element['limit']
    calc_el_symbol = calc_element['el']

    # 2. Define Helper for Search (Recursive)
    def search_formulas(target_neutral_mass):
        # Calculate tolerance in Da
        if tol_unit == "ppm":
            tolerance_da = target_neutral_mass * tol / 1e6
        else:
            tolerance_da = tol
        
        candidates = []
        
        def recurse(level, current_mass, current_counts):
            if current_mass > target_neutral_mass + tolerance_da:
                return

            if level == len(iteration_elements):
                remaining_mass = target_neutral_mass - current_mass
                count_est = remaining_mass / calc_el_mass
                count = int(round(count_est))
                
                if count < 0:
                    return
                if count > calc_el_limit:
                    return
                
                final_mass = current_mass + count * calc_el_mass
                diff = abs(final_mass - target_neutral_mass)
                
                if diff <= tolerance_da:
                    full_counts = {calc_el_symbol: count}
                    for i, el_info in enumerate(iteration_elements):
                        full_counts[el_info['el']] = current_counts[i]
                    
                    # --- Filtering ---
                    
                    c_count = full_counts.get('C', 0)
                    h_count = full_counts.get('H', 0)
                    n_count = full_counts.get('N', 0)
                    o_count = full_counts.get('O', 0)

                    # 1. H/C Ratio
                    if apply_hc_ratio:
                        if c_count > 0:
                            if h_count / c_count > 4:
                                return
                        if c_count == 0 and h_count > 0:
                            return
                    
                    # 2. Nitrogen Rule
                    if apply_nitrogen_rule:
                        calc_el_nom_mass = full_counts.get(calc_el_symbol, 0) * calc_element['nom_mass']
                        iter_el_nom_mass = sum(full_counts[el_info['el']] * el_info['nom_mass'] for el_info in iteration_elements)
                        nom_mass = calc_el_nom_mass + iter_el_nom_mass
                        if not (nom_mass % 2 == n_count % 2):
                            return

                    # 3. RDBE filter (rings + double bond equivalents)
                    # RDBE = C - H/2 + N/2 + 1. Ignores O/S (even-valence, no effect on RDBE).
                    # Negative RDBE is chemically impossible -> hard reject regardless of rdbe_min
                    # unless caller explicitly widens rdbe_min below 0.
                    if apply_rdbe_filter:
                        if c_count == 0:
                            # No carbon skeleton -> RDBE concept doesn't meaningfully apply;
                            # treat as implausible for organic aerosol formulas.
                            return
                        rdbe_val = c_count - h_count / 2 + n_count / 2 + 1
                        if rdbe_val < rdbe_min or rdbe_val > rdbe_max:
                            return

                    # 4. Heteroatom ratio filter (O/C, N/C bounds)
                    if apply_ratio_filter:
                        if c_count == 0:
                            return
                        if (o_count / c_count) > oc_max:
                            return
                        if (n_count / c_count) > nc_max:
                            return

                    all_elements_keys = sorted(full_counts.keys())
                    sorted_keys = []
                    if 'C' in all_elements_keys:
                        sorted_keys.append('C')
                        all_elements_keys.remove('C')
                    if 'H' in all_elements_keys:
                        sorted_keys.append('H')
                        all_elements_keys.remove('H')
                    sorted_keys.extend(all_elements_keys)
                    
                    final_counts_list = [full_counts[k] for k in sorted_keys]
                    
                    candidates.append({
                        'formula_str': get_formula_string(final_counts_list, elements=sorted_keys),
                        'mass': final_mass,
                        'error': diff
                    })
                return

            el_info = iteration_elements[level]
            el_mass = el_info['mass']
            el_limit = el_info['limit']
            
            for c in range(el_limit + 1):
                new_mass = current_mass + c * el_mass
                if new_mass > target_neutral_mass + tolerance_da:
                    break
                recurse(level + 1, new_mass, current_counts + [c])

        recurse(0, 0.0, [])
        return candidates

    # 3. Main Logic with Fallbacks
    
    primary_neutral_mass = None
    if mode == "positive" or mode =="pos":
        primary_neutral_mass = target - mass_proton
    elif mode == "negative" or mode =="neg":
        primary_neutral_mass = target + mass_proton
    else:
        primary_neutral_mass = target
        
    results = search_formulas(primary_neutral_mass)
    
    if results:
        results.sort(key=lambda x: x['error'])
        best = results[0]
        if not return_tuple:
            return best['formula_str'], best['mass']
        else:            
           return (best['formula_str'], best['mass'])
        
    fallback_neutral_mass = None
    
    if mode == "positive" or mode =="pos":
        mz_sodium_ion = mass_Na - mass_electron
        fallback_neutral_mass = target - mz_sodium_ion
        
        results_na = search_formulas(fallback_neutral_mass)
        if results_na:
            results_na.sort(key=lambda x: x['error'])
            best = results_na[0]
            if not return_tuple:
                return best['formula_str'], best['mass']
            else:            
                return (best['formula_str'], best['mass'])
            
    elif mode == "negative" or mode =="neg":
        fallback_neutral_mass = (target + mass_proton) / 2.0
        
        results_dimer = search_formulas(fallback_neutral_mass)
        if results_dimer:
            results_dimer.sort(key=lambda x: x['error'])
            best = results_dimer[0]
            if not return_tuple:
                return best['formula_str'], best['mass']
            else:            
                return (best['formula_str'], best['mass'])
            
    return no_match_return, np.nan

def get_formula_string(counts ,elements=["C", "H", "N", "O", "S"]):
    formula_parts = []
    for el, n in zip(elements, counts):
        if n == 1:
            formula_parts.append(el)
        elif n > 1:
            formula_parts.append(f"{el}{n}")
    return ''.join(formula_parts)

In [3]:
# Example:  caffeine in positive mode [M+H]+, C8H11N4O2 
# caffeine itself is 194.08037557 at neutral, C8H10N4O2

target_mz =  195.0882 # caffeine in positive mode

formula, target_neutral_mass = compute_formula_from_mass_efficient(target_mz, mode="pos")
print( formula, target_neutral_mass)


target_mz =  195.0870 # assuming a slightly off mass of caffeine, 

formula, target_neutral_mass = compute_formula_from_mass_efficient(target_mz, mode="pos")
print( formula, target_neutral_mass)

C8H10N4O2 194.080376319
C8H10N4O2 194.080376319
